## SST dataset downloader

In [1]:
import os
import numpy as np
import xarray as xr
import urllib.request
import urllib.error
import time
import http.client
import socket
import re
import glob


## 0- General parameters
DATA_DIR = "datasets/Data_sst"
DATA_DIR_CLEAN = "datasets/Data_sst_clean"
FINAL_DATA_DIR = "datasets"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(DATA_DIR_CLEAN, exist_ok=True)
BASE_URL = "https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2.highres/"

## 1- Download parameters
START_YEAR = 2000
END_YEAR = 2025
MASK_FILE = "lsmask.oisst.nc"
YEAR_FILE_TEMPLATE = "sst.day.mean.{year}.nc"
YEAR_FILES = [f"sst.day.mean.{y}.nc" for y in range(START_YEAR, END_YEAR + 1)]

## 2- Resize parameters
DOWNSAMPLE_FACTOR = 4       #square 4*4 pixels -> 1 pixel

## 1- Download

In [2]:
def _remote_size(url, timeout=20):
    try:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; sst-downloader/1.0)"}
        request = urllib.request.Request(url, method="HEAD", headers=headers)
        with urllib.request.urlopen(request, timeout=timeout) as response:
            size = response.headers.get("Content-Length")
        return int(size) if size is not None else None
    except Exception as exc:
        print(f"  could not determine remote size via HEAD ({exc}); will size from GET instead.")
        return None

def _download(filename, max_retries=200, timeout=20):
    path = os.path.join(DATA_DIR, filename)
    url = BASE_URL + filename
    headers = {"User-Agent": "Mozilla/5.0 (compatible; sst-downloader/1.0)"}

    expected_size = _remote_size(url)
    if os.path.exists(path) and expected_size is not None and os.path.getsize(path) == expected_size:
        print(f"{filename} already present, skipping download.")
        return path

    # if exists already in Data_cleaned, skip download
    year_match = re.search(r"(\d{4})", filename)
    path_cleaned = os.path.join(DATA_DIR_CLEAN, f"sst_{year_match.group(1)}.npy") if year_match else None
    if path_cleaned and os.path.exists(path_cleaned):
        print(f"{filename} already processed and present in cleaned data, skipping download.")
        return path_cleaned

    tmp_path = os.path.join(DATA_DIR, f".{filename}.part")
    attempt = 0

    while True:
        attempt += 1
        resume_from = os.path.getsize(tmp_path) if os.path.exists(tmp_path) else 0

        if expected_size is not None and resume_from >= expected_size:
            break  # done

        if attempt > max_retries:
            raise IOError(
                f"Gave up after {max_retries} attempts: "
                f"{resume_from} out of {expected_size} bytes for {filename}"
            )

        req_headers = dict(headers)
        mode = "wb"
        if resume_from > 0:
            req_headers["Range"] = f"bytes={resume_from}-"
            mode = "ab"

        pct = (resume_from / expected_size * 100) if expected_size else 0
        print(f"[attempt {attempt}] {filename}: {resume_from}/{expected_size} bytes ({pct:.1f}%)")

        try:
            request = urllib.request.Request(url, headers=req_headers)
            with urllib.request.urlopen(request, timeout=timeout) as response:
                if resume_from > 0 and response.status == 200:
                    # server ignored Range, restart from scratch
                    resume_from = 0
                    mode = "wb"

                with open(tmp_path, mode) as target:
                    while True:
                        try:
                            chunk = response.read(1024 * 256)
                        except (socket.timeout, http.client.IncompleteRead) as e:
                            raise IOError(f"stream interrupted: {e}")
                        if not chunk:
                            break
                        target.write(chunk)

            # Response body fully consumed without error.
            if expected_size is None:
                # We didn't know the size upfront (HEAD failed); trust that a
                # clean read-to-EOF means the transfer completed successfully.
                expected_size = os.path.getsize(tmp_path)
            break

        except (urllib.error.URLError, socket.timeout, http.client.IncompleteRead,
                ConnectionResetError, TimeoutError, OSError) as exc:
            wait = min(2 + attempt // 5, 15)
            print(f"  interrupted: {exc}. Resuming in {wait}s...")
            time.sleep(wait)
            continue

    got = os.path.getsize(tmp_path)
    if expected_size is not None and got != expected_size:
        raise IOError(f"retrieval incomplete after loop: got {got} out of {expected_size} bytes")

    os.replace(tmp_path, path)
    print(f"{filename} download complete.")
    return path

_download(MASK_FILE)  
for fname in YEAR_FILES:
    F = _download(fname)
    del F

lsmask.oisst.nc already present, skipping download.
sst.day.mean.2000.nc already present, skipping download.
sst.day.mean.2001.nc already present, skipping download.
sst.day.mean.2002.nc already present, skipping download.
sst.day.mean.2003.nc already present, skipping download.
sst.day.mean.2004.nc already present, skipping download.
sst.day.mean.2005.nc already present, skipping download.
sst.day.mean.2006.nc already present, skipping download.
sst.day.mean.2007.nc already present, skipping download.
sst.day.mean.2008.nc already present, skipping download.
sst.day.mean.2009.nc already present, skipping download.
sst.day.mean.2010.nc already present, skipping download.
sst.day.mean.2011.nc already present, skipping download.
sst.day.mean.2012.nc already present, skipping download.
sst.day.mean.2013.nc already present, skipping download.
sst.day.mean.2014.nc already present, skipping download.
sst.day.mean.2015.nc already present, skipping download.
sst.day.mean.2016.nc already present

## 2 - Resize 

In [3]:
def _load_and_modify_year_matrix(year_file, mask_file, verbose=False):

    # data extraction
    year_path = os.path.join(DATA_DIR, year_file)
    mask_path = os.path.join(DATA_DIR, mask_file)
    ds_year = xr.open_dataset(year_path)
    ds_mask = xr.open_dataset(mask_path)
    data = ds_year["sst"].values         # (time, lat, lon)
    mask = ds_mask["lsmask"].values      # (time, lat, lon) or (lat, lon)

    if verbose:
        print(f"raw shape: {data.shape}")
    ds_year.close()
    ds_mask.close()

    # The mask file includes a singleton leading time dimension; squeeze it to 2D.
    mask = np.squeeze(mask)
    if mask.ndim != 2:
        raise ValueError(f"Expected a 2D mask after squeezing, got shape {mask.shape}")
    if mask.shape != data.shape[1:]:
        raise ValueError(
            f"Mask spatial shape {mask.shape} does not match SST spatial shape {data.shape[1:]}"
        )

    # Subsample the data by averaging 4x4 blocks to reduce resolution
    data = data.reshape(
        data.shape[0],
        data.shape[1] // DOWNSAMPLE_FACTOR,
        DOWNSAMPLE_FACTOR,
        data.shape[2] // DOWNSAMPLE_FACTOR,
        DOWNSAMPLE_FACTOR,
    ).mean(axis=(2, 4))
    mask = mask.reshape(
        mask.shape[0] // DOWNSAMPLE_FACTOR,
        DOWNSAMPLE_FACTOR,
        mask.shape[1] // DOWNSAMPLE_FACTOR,
        DOWNSAMPLE_FACTOR,
    ).mean(axis=(1, 3)) > 0.5

    # Handle NaN values and flip latitude axis for visualization consistency
    data = np.nan_to_num(data, nan=0.0)
    data[:, ~mask] = 0.0            # Apply land-sea mask: set land pixels to 0
    data = np.flip(data, axis=1)    # Flip latitude axis for visualization consistency

    if verbose:
        print(f"shape after averaging: {data.shape}")
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 5))
        for i in range(5):
            plt.subplot(1, 5, i + 1)
            plt.imshow(data[i*50], cmap='viridis')
            plt.title(f"Day {i*50+1}")
            plt.axis('off')
        plt.show()

    data = data.reshape(data.shape[0], -1)  # (time, lat*lon)
    del ds_year, ds_mask
    return data

def _resize_all_years():

    for year in range(START_YEAR, END_YEAR + 1):
        year_file = YEAR_FILE_TEMPLATE.format(year=year)
        year_path = os.path.join(DATA_DIR, year_file)

        if not os.path.exists(year_path):
            print(f"Skipping {year} already processed -> {year_path}")
            continue

        data = _load_and_modify_year_matrix(year_file, MASK_FILE)
        # remove the original file to save space
        os.remove(year_path)
        print(f"Removed original file: {year_path}")
        output_path = os.path.join(DATA_DIR_CLEAN, f"sst_{year}.npy")
        np.save(output_path, data)
        print(f"Saved the resized data: {output_path} | shape={data.shape}")

_resize_all_years()

Removed original file: datasets/Data_sst/sst.day.mean.2000.nc
Saved the resized data: datasets/Data_sst_clean/sst_2000.npy | shape=(366, 64800)
Removed original file: datasets/Data_sst/sst.day.mean.2001.nc
Saved the resized data: datasets/Data_sst_clean/sst_2001.npy | shape=(365, 64800)
Removed original file: datasets/Data_sst/sst.day.mean.2002.nc
Saved the resized data: datasets/Data_sst_clean/sst_2002.npy | shape=(365, 64800)
Removed original file: datasets/Data_sst/sst.day.mean.2003.nc
Saved the resized data: datasets/Data_sst_clean/sst_2003.npy | shape=(365, 64800)
Removed original file: datasets/Data_sst/sst.day.mean.2004.nc
Saved the resized data: datasets/Data_sst_clean/sst_2004.npy | shape=(366, 64800)
Removed original file: datasets/Data_sst/sst.day.mean.2005.nc
Saved the resized data: datasets/Data_sst_clean/sst_2005.npy | shape=(365, 64800)
Removed original file: datasets/Data_sst/sst.day.mean.2006.nc
Saved the resized data: datasets/Data_sst_clean/sst_2006.npy | shape=(365,

## 3 - Unify

In [4]:
def _unify():
    files = sorted(glob.glob(os.path.join(DATA_DIR_CLEAN, "*.npy")))

    arrays = []
    first_shape = None

    for f in files:
        arr = np.load(f)
        if arr.ndim != 2:
            raise ValueError(f"Expected 2D (time, data) array in {f}, got shape {arr.shape}")

        if first_shape is None:
            first_shape = arr.shape
        elif arr.shape[1] != first_shape[1]:
            raise ValueError(
                f"Feature dimension mismatch in {f}: expected {first_shape[1]}, got {arr.shape[1]}"
            )

        arrays.append(arr)
        print(f"Loaded {os.path.basename(f)} with shape {arr.shape}, total time steps so far: {sum(a.shape[0] for a in arrays)}")

    # Stack all years along time axis: (sum_time, data)
    data = np.concatenate(arrays, axis=0)
    print(f"Final stacked data shape: {data.shape}")
    # save the stacked data to a new .npy file in the  final data directory
    output_file = os.path.join(FINAL_DATA_DIR, "sst_all.npy")
    np.save(output_file, data)

_unify()

Loaded sst_2000.npy with shape (366, 64800), total time steps so far: 366
Loaded sst_2001.npy with shape (365, 64800), total time steps so far: 731
Loaded sst_2002.npy with shape (365, 64800), total time steps so far: 1096
Loaded sst_2003.npy with shape (365, 64800), total time steps so far: 1461
Loaded sst_2004.npy with shape (366, 64800), total time steps so far: 1827
Loaded sst_2005.npy with shape (365, 64800), total time steps so far: 2192
Loaded sst_2006.npy with shape (365, 64800), total time steps so far: 2557
Loaded sst_2007.npy with shape (365, 64800), total time steps so far: 2922
Loaded sst_2008.npy with shape (366, 64800), total time steps so far: 3288
Loaded sst_2009.npy with shape (365, 64800), total time steps so far: 3653
Loaded sst_2010.npy with shape (365, 64800), total time steps so far: 4018
Loaded sst_2011.npy with shape (365, 64800), total time steps so far: 4383
Loaded sst_2012.npy with shape (366, 64800), total time steps so far: 4749
Loaded sst_2013.npy with sh

## 3b - ROM Unify

In [5]:
def _ROM_unify():
    files = sorted(glob.glob(os.path.join(DATA_DIR_CLEAN, "*.npy")))

    yearly_arrays = []
    expected_features = None
    target_days = 365

    for f in files:
        arr = np.load(f)
        if arr.ndim != 2:
            raise ValueError(f"Expected 2D (time, data) array in {f}, got shape {arr.shape}")

        if expected_features is None:
            expected_features = arr.shape[1]
        elif arr.shape[1] != expected_features:
            raise ValueError(
                f"Feature dimension mismatch in {f}: expected {expected_features}, got {arr.shape[1]}"
            )

        if arr.shape[0] == target_days + 1:
            arr = arr[:target_days]
            print(f"Loaded {os.path.basename(f)} with shape {arr.shape} after dropping leap day")
        elif arr.shape[0] != target_days:
            raise ValueError(
                f"Expected {target_days} or {target_days + 1} time steps in {f}, got {arr.shape[0]}")
        else:
            print(f"Loaded {os.path.basename(f)} with shape {arr.shape}")

        yearly_arrays.append(arr)
        #print(f"Years loaded so far: {len(yearly_arrays)}")

    # Stack all years into (year, 365, spatial)
    data = np.stack(yearly_arrays, axis=0)
    print(f"Final stacked ROM data shape: {data.shape}")
    # save the stacked data to a new .npy file in the final data directory
    output_file = os.path.join(FINAL_DATA_DIR, "sst_all_ROM.npy")
    np.save(output_file, data)

_ROM_unify()

Loaded sst_2000.npy with shape (365, 64800) after dropping leap day
Loaded sst_2001.npy with shape (365, 64800)
Loaded sst_2002.npy with shape (365, 64800)
Loaded sst_2003.npy with shape (365, 64800)
Loaded sst_2004.npy with shape (365, 64800) after dropping leap day
Loaded sst_2005.npy with shape (365, 64800)
Loaded sst_2006.npy with shape (365, 64800)
Loaded sst_2007.npy with shape (365, 64800)
Loaded sst_2008.npy with shape (365, 64800) after dropping leap day
Loaded sst_2009.npy with shape (365, 64800)
Loaded sst_2010.npy with shape (365, 64800)
Loaded sst_2011.npy with shape (365, 64800)
Loaded sst_2012.npy with shape (365, 64800) after dropping leap day
Loaded sst_2013.npy with shape (365, 64800)
Loaded sst_2014.npy with shape (365, 64800)
Loaded sst_2015.npy with shape (365, 64800)
Loaded sst_2016.npy with shape (365, 64800) after dropping leap day
Loaded sst_2017.npy with shape (365, 64800)
Loaded sst_2018.npy with shape (365, 64800)
Loaded sst_2019.npy with shape (365, 64800)


## 4 - Remove the data directories

In [6]:
# remove the foler Data_sst and Data_sst_clean to save space
import shutil
#shutil.rmtree(DATA_DIR)
#shutil.rmtree(DATA_DIR_CLEAN)

## 5 - Buoy data


In [3]:
"""Seleziona 20 boe NDBC ben distribuite con disponibilità WTMP >= 40%.

Disponibilità = (giorni con almeno una misura valida di WTMP) / (365 * n_anni).
Usa la stessa cache dello script principale (datasets/Data_buoy/_cache).
"""
import gzip
import io
import time
from collections import defaultdict
from itertools import zip_longest
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://www.ndbc.noaa.gov/data/historical/stdmet/{station}h{year}.txt.gz"
CACHE_DIR = Path("datasets/Data_buoy/_cache")
YEARS = range(2000, 2026)
MIN_AVAIL = 0.40
N_PICK = 20
MISSING_THRESHOLD = 90

# Candidati (ID scritti a memoria: quelli inesistenti vengono semplicemente scartati)
CANDIDATES = [
    "46001", "46035", "46066", "46073", "46006", "46002", "46005", "46059",
    "46013", "46026", "46025", "46042", "46011", "46086", "46047",
    "51000", "51001", "51002", "51003", "51004", "51028",
    "52200", "52211", "32012",
    "13001", "13002", "13008",
    "41001", "41002", "41004", "41010", "41013", "41043", "41044", "41046",
    "42001", "42002", "42019", "42040", "42056", "42057", "42058", "42059",
    "44008", "44011", "44013", "44014", "44025", "44139", "44251",
    "62001", "62029", "23001", "23008",
]


def valid_days(station: str, year: int) -> int | None:
    """Giorni con WTMP valido; None se il file non esiste."""
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = CACHE_DIR / f"{station}_{year}.txt.gz"
    if cache_file.exists():
        raw = cache_file.read_bytes()
    else:
        try:
            r = requests.get(BASE_URL.format(station=station, year=year), timeout=60)
        except requests.RequestException:
            return None
        if r.status_code != 200:
            return None
        raw = r.content
        cache_file.write_bytes(raw)
        time.sleep(0.3)
    try:
        lines = gzip.decompress(raw).decode("utf-8", errors="replace").splitlines()
        header = lines[0].lstrip("#").split()
        body = [ln for ln in lines[1:] if not ln.startswith("#")]
        df = pd.read_csv(io.StringIO("\n".join(body)), sep=r"\s+", names=header, header=None)
        if "WTMP" not in df.columns:
            return 0
        ycol = "YYYY" if "YYYY" in df.columns else "YY"
        y = df[ycol].astype(int)
        y = y.where(y >= 100, y + 2000)
        w = pd.to_numeric(df["WTMP"], errors="coerce")
        w = w.where(w < MISSING_THRESHOLD)
        d = pd.to_datetime(dict(year=y, month=df["MM"], day=df["DD"]), errors="coerce")
        return int(d[w.notna()].dt.normalize().nunique())
    except Exception:
        return None


rows = []
for st in CANDIDATES:
    tot = sum(valid_days(st, y) or 0 for y in YEARS)
    avail = tot / (365 * len(YEARS))
    rows.append((st, avail))
    print(f"{st}: {avail:6.1%}")

ok = sorted([r for r in rows if r[1] >= MIN_AVAIL], key=lambda r: -r[1])

# Distribuzione: round-robin tra gruppi di prefisso (≈ regione/rete), migliori per primi
groups = defaultdict(list)
for st, a in ok:
    groups[st[:2]].append((st, a))
picked = []
for tier in zip_longest(*groups.values()):
    for item in tier:
        if item and len(picked) < N_PICK:
            picked.append(item)

print(f"\nIdonee (>= {MIN_AVAIL:.0%}): {len(ok)}")
print(f"\nDEFAULT_STATIONS = {[s for s, _ in picked]}")
if len(picked) < N_PICK:
    print(f"ATTENZIONE: solo {len(picked)} stazioni sopra soglia; abbassa MIN_AVAIL o aggiungi candidati.")

46001:  90.3%
46035:  70.4%
46066:  71.4%
46073:  50.8%
46006:  59.5%
46002:  66.3%
46005:  61.8%
46059:  68.7%
46013:  81.6%
46026:  92.1%
46025:  95.4%
46042:  91.3%
46011:  89.9%
46086:  78.0%
46047:  74.7%
51000:  62.7%
51001:  70.0%
51002:  75.3%
51003:  78.5%
51004:  86.0%
51028:  31.1%
52200:  71.2%
52211:  30.2%
32012:   0.0%
13001:   0.0%
13002:   0.0%
13008:   0.0%
41001:  57.1%
41002:  69.2%
41004:  89.6%
41010:  81.5%
41013:  80.0%
41043:  65.7%
41044:  57.4%
41046:  62.4%
42001:  81.2%
42002:  85.7%
42019:  74.8%
42040:  80.5%
42056:  68.5%
42057:  53.1%
42058:  56.3%
42059:  48.1%
44008:  77.8%
44011:  68.4%
44013:  97.4%
44014:  86.2%
44025:  89.5%
44139:   0.0%
44251:   0.0%
62001:   0.0%
62029:   0.0%
23001:   0.0%
23008:   0.0%

Idonee (>= 40%): 42

DEFAULT_STATIONS = ['44013', '46025', '41004', '51004', '42002', '52200', '44025', '46026', '41010', '51003', '42001', '44014', '46042', '41013', '51002', '42040', '44008', '46001', '41002', '51001']


In [4]:
import argparse
import gzip
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

BASE_URL = "https://www.ndbc.noaa.gov/data/historical/stdmet/{station}h{year}.txt.gz"

# Edit this list with the buoys you are interested in
# (station list: https://www.ndbc.noaa.gov/to_station.shtml)
DEFAULT_STATIONS = [
    # Pacifico nord (9)
    "46035",  # Mare di Bering        70.4%
    "46001",  # Golfo dell'Alaska     90.3%
    "46066",  # Pacifico nord         71.4%
    "46006",  # Pacifico NE, offshore 59.5%
    "46059",  # Pacifico NE           68.7%
    "46025",  # Santa Monica Basin    95.4%
    "51001",  # Hawaii nord-ovest     70.0%
    "51004",  # Hawaii sud-est        86.0%
    "52200",  # Guam                  71.2%
    # Atlantico / Caraibi / Golfo (11)
    "44013",  # Boston                97.4%
    "41001",  # Hatteras offshore     57.1%
    "41010",  # Canaveral             81.5%
    "41046",  # Bahamas est           62.4%
    "41043",  # Nord Porto Rico       65.7%
    "41044",  # Atlantico centro-nord 57.4%
    "42057",  # Caraibi ovest         53.1%
    "42058",  # Caraibi centrali      56.3%
    "42040",  # Golfo nord            80.5%
    "42001",  # Golfo centrale        81.2%
    "42002",  # Golfo ovest           85.7%
]

OUT_DIR = "datasets/Data_buoy"

MISSING_THRESHOLD = 90  # NDBC uses 99.0 / 999.0 as the missing-value flag


def download_station_year(station: str, year: int, cache_dir: Path) -> pd.Series | None:
    """Download (with cache) the yearly file of a station and return the daily mean of WTMP."""
    cache_file = cache_dir / f"{station}_{year}.txt.gz"

    if cache_file.exists():
        raw = cache_file.read_bytes()
    else:
        url = BASE_URL.format(station=station, year=year)
        try:
            r = requests.get(url, timeout=60)
        except requests.RequestException as e:
            print(f"  [network error] {station} {year}: {e}")
            return None
        if r.status_code == 404:
            return None  # station not active in that year
        if r.status_code != 200:
            print(f"  [HTTP {r.status_code}] {station} {year}")
            return None
        raw = r.content
        cache_file.write_bytes(raw)
        time.sleep(0.3)  # be gentle with the server

    try:
        text = gzip.decompress(raw).decode("utf-8", errors="replace")
    except OSError:
        print(f"  [corrupt file] {station} {year}")
        cache_file.unlink(missing_ok=True)
        return None

    lines = text.splitlines()
    if not lines:
        return None

    # First line = header (starts with '#'); since 2007 the second line holds the units
    header = lines[0].lstrip("#").split()
    data_lines = [ln for ln in lines[1:] if not ln.startswith("#")]
    df = pd.read_csv(io.StringIO("\n".join(data_lines)), sep=r"\s+", names=header, header=None)

    if "WTMP" not in df.columns:
        return None

    # Normalize the date columns (YY or YYYY)
    year_col = "YYYY" if "YYYY" in df.columns else "YY"
    df["year"] = df[year_col].astype(int)
    df.loc[df["year"] < 100, "year"] += 2000
    df["date"] = pd.to_datetime(
        dict(year=df["year"], month=df["MM"], day=df["DD"]), errors="coerce"
    )

    df["WTMP"] = pd.to_numeric(df["WTMP"], errors="coerce")
    df.loc[df["WTMP"] >= MISSING_THRESHOLD, "WTMP"] = float("nan")

    return df.groupby("date")["WTMP"].mean().round(2)


def build_matrix(year: int, stations: list[str], cache_dir: Path) -> pd.DataFrame:
    """Matrix (days x stations) for one year: 365 rows, leap years without Dec 31."""
    days = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
    if len(days) == 366:
        days = days[:-1]  # drop the last day in leap years

    matrix = pd.DataFrame(np.nan, index=days, columns=stations, dtype=float)

    for st in stations:
        series = download_station_year(st, year, cache_dir)
        if series is None or series.empty:
            print(f"  {st}: no data")
            continue
        matrix[st] = series.reindex(days)
        print(f"  {st}: {int(matrix[st].notna().sum())} days with data")

    matrix.index = range(1, len(matrix) + 1)  # day of year 1..365
    matrix.index.name = "day"
    return matrix


def _buoy_main():
    p = argparse.ArgumentParser(
        description="Yearly sea water temperature matrices from NOAA NDBC buoys"
    )
    p.add_argument("--stations", nargs="+", default=DEFAULT_STATIONS, help="NDBC station IDs")
    p.add_argument("--start", type=int, default=2000, help="first year")
    p.add_argument("--end", type=int, default=2025, help="last year")
    p.add_argument("--out", default=OUT_DIR, help="output directory")
    # parse_known_args ignores extra arguments (e.g. the --f=... passed by Jupyter/IPython)
    args, _ = p.parse_known_args()

    out_dir = Path(args.out)
    cache_dir = out_dir / "_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    years = list(range(args.start, args.end + 1))
    yearly = []

    for year in years:
        print(f"Year {year}")
        matrix = build_matrix(year, args.stations, cache_dir)
        assert matrix.shape[0] == 365
        arr = matrix.to_numpy(dtype=np.float64)  # missing data = NaN
        np.save(out_dir / f"matrix_{year}.npy", arr)
        yearly.append(arr)
        print(f"  -> saved matrix_{year}.npy (shape {arr.shape[0]} x {arr.shape[1]})")

    # single array with everything: (n_years, 365, n_sensors)
    all_data = np.stack(yearly, axis=0)
    np.save(out_dir / "buoy_all_ROM.npy", all_data)
    print(f"-> saved matrix_all.npy (shape {all_data.shape})")

    # labels for the axes
    (out_dir / "years.txt").write_text("\n".join(map(str, years)) + "\n")
    (out_dir / "stations.txt").write_text("\n".join(args.stations) + "\n")

    print("Done.")


_buoy_main()

Year 2000
  46035: 365 days with data
  46001: 365 days with data
  46066: 173 days with data
  46006: 134 days with data
  46059: 365 days with data
  46025: 365 days with data
  51001: 365 days with data
  51004: 365 days with data
  52200: no data
  44013: 365 days with data
  41001: 192 days with data
  41010: 365 days with data
  41046: no data
  41043: no data
  41044: no data
  42057: no data
  42058: no data
  42040: 341 days with data
  42001: 365 days with data
  42002: 331 days with data
  -> saved matrix_2000.npy (shape 365 x 20)
Year 2001
  46035: 76 days with data
  46001: 365 days with data
  46066: 364 days with data
  46006: 365 days with data
  46059: 338 days with data
  46025: 363 days with data
  51001: 365 days with data
  51004: 365 days with data
  52200: no data
  44013: 365 days with data
  41001: 365 days with data
  41010: 365 days with data
  41046: no data
  41043: no data
  41044: no data
  42057: no data
  42058: no data
  42040: 365 days with data
  420

## 6 - Sensor locations

In [5]:
import argparse
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import requests

ACTIVE_URL = "https://www.ndbc.noaa.gov/activestations.xml"
PAGE_URL = "https://www.ndbc.noaa.gov/station_page.php?station={station}"
COORD_RE = re.compile(r"(\d+\.\d+)\s*([NS])\s+(\d+\.\d+)\s*([EW])")


def load_active_stations() -> dict:
    """Return {station_id: (lat, lon)} from the NDBC active-station list."""
    try:
        r = requests.get(ACTIVE_URL, timeout=60)
        r.raise_for_status()
        root = ET.fromstring(r.content)
    except (requests.RequestException, ET.ParseError) as e:
        print(f"[warning] could not download the active-station list: {e}")
        return {}

    out = {}
    for s in root.iter("station"):
        try:
            out[s.get("id").upper()] = (float(s.get("lat")), float(s.get("lon")))
        except (TypeError, ValueError):
            continue
    return out


def from_station_page(station: str) -> tuple[float, float] | None:
    """Fallback: read coordinates from the station page (e.g. '33.749 N 119.053 W')."""
    try:
        r = requests.get(PAGE_URL.format(station=station), timeout=60)
        r.raise_for_status()
    except requests.RequestException:
        return None
    m = COORD_RE.search(r.text)
    if not m:
        return None
    lat = float(m.group(1)) * (1 if m.group(2) == "N" else -1)
    lon = float(m.group(3)) * (1 if m.group(4) == "E" else -1)
    return lat, lon


def main():
    p = argparse.ArgumentParser(description="Latitude/longitude of NOAA NDBC buoys")
    p.add_argument("--stations", nargs="+", default=None, help="NDBC station IDs")
    p.add_argument("--out", default="datasets", help="output directory")
    # parse_known_args ignores extra arguments (e.g. the --f=... passed by Jupyter/IPython)
    args, _ = p.parse_known_args()

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    if args.stations:
        stations = args.stations
    elif (out_dir / "stations.txt").exists():
        stations = (out_dir / "stations.txt").read_text().split()
    else:
        stations = DEFAULT_STATIONS

    active = load_active_stations()

    rows = []
    print(f"{'station':<8} {'lat':>9} {'lon':>10}  source")
    for st in stations:
        pos, source = active.get(st.upper()), "active list"
        if pos is None:
            pos, source = from_station_page(st), "station page"
        if pos is None:
            print(f"{st:<8} {'n/a':>9} {'n/a':>10}  not found")
            rows.append(f"{st},,,not found")
            continue
        lat, lon = pos
        print(f"{st:<8} {lat:>9.3f} {lon:>10.3f}  {source}")
        rows.append(f"{st},{lat},{lon},{source}")

    out_file = out_dir / "stations_location.csv"
    out_file.write_text("station,lat,lon,source\n" + "\n".join(rows) + "\n")
    print(f"-> saved {out_file}")


if __name__ == "__main__":
    main()

station        lat        lon  source
46035       57.034   -177.468  active list
46001       56.296   -148.027  active list
46066       52.776   -154.992  active list
46006       40.730   -137.420  active list
46059       38.067   -129.895  active list
46025       33.765   -119.077  active list
51001       24.475   -162.030  active list
51004       17.504   -152.197  active list
52200       13.354    144.788  active list
44013       42.346    -70.651  active list
41001       34.738    -72.503  active list
41010       28.860    -78.478  active list
41046       23.840    -68.340  active list
41043       21.090    -64.864  active list
41044       21.582    -58.630  active list
42057       16.975    -81.578  active list
42058       14.114    -75.949  active list
42040       29.207    -88.237  active list
42001       25.922    -89.638  active list
42002       25.950    -93.780  active list
-> saved datasets/stations_location.csv
